# ARIA — Transformer Model Fine-Tuning Pipeline (Google Colab T4 GPU)

**Purpose**: Fine-tunes two separate pre-trained transformer models for the ARIA SAP Support Ticket Intelligence System:
1. **DistilBERT (Binary Classifier)**: Classifies incoming user tickets as either an `incident` (0) or `development` request (1).
2. **BERT (Multi-Label Classifier)**: Simultaneously predicts `module` (7 classes), `issue_type` (8 classes), and `complexity` (3 classes) for incident tickets via multi-task learning.

**Dataset**: Trained on `incoming_tickets_clean.csv` produced by `inspect_data.ipynb`.

**Data Split Strategy**:
- **Training Set (80%)**: Data the model learns weights from.
- **Validation Set (10%)**: Monitored during training for early checkpointing and overfitting detection — never trained on.
- **Test Set (10%)**: Completely hidden during training; evaluated once at the end for unbiased performance metrics.

*Note: All split sizes are computed dynamically from `len(df)` to support variable dataset lengths.*

## 1.2 — Install Required Dependencies

**Why this cell exists**: Installs HuggingFace `transformers` for pre-trained model architectures, `datasets` for data utilities, `torch` for PyTorch deep learning backend, `scikit-learn` for metrics and stratified splitting, and visualization packages.

In [ ]:
# Quietly install all required machine learning and NLP packages
!pip install -q transformers datasets torch scikit-learn pandas numpy seaborn matplotlib tqdm

print("All required dependencies successfully installed.")

## 1.3 — Hardware Acceleration Check (GPU Verification)

**Why this cell exists**: Verifies that a GPU (e.g., NVIDIA T4) is attached to the Colab runtime and reports available Video RAM.

In [ ]:
import torch

# Determine if CUDA GPU acceleration is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Please change runtime type: Runtime > Change runtime type > T4 GPU")

## 1.4 — Global Imports and Deterministic Seed Initialization

**Why this cell exists**: Imports python modules and fixes all pseudo-random number generator seeds to `42` ensuring 100% reproducible training runs across different executions.

In [ ]:
# Import core python, data manipulation, and PyTorch modules
import os
import sys
import json
import math
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Set seaborn visual styling
sns.set_theme(style="darkgrid")

# Set deterministic seeds for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Global random seeds set to {SEED} for full reproducibility.")

# SECTION 2 — DATA LOADING AND PREPARATION

## 2.1 — Upload and Load Clean Dataset

**Why this cell exists**: Prompts the user to upload `incoming_tickets_clean.csv` directly in Colab and loads it into a pandas DataFrame using dynamic row counting.

In [ ]:
# Prompt user to upload incoming_tickets_clean.csv
from google.colab import files
print("Please select and upload 'incoming_tickets_clean.csv':")
uploaded = files.upload()

# Variable definition for file path
CSV_FILENAME = "incoming_tickets_clean.csv"
df = pd.read_csv(CSV_FILENAME)

print(f"\nTotal rows loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print("\nTicket type distribution:")
print(df["ticket_type"].value_counts())
print(f"\nNote: Actual row count after cleaning is {len(df):,}. All split sizes will derive dynamically.")
df.head(3)

## 2.2 — Comprehensive Exploratory Data Audit

**Why this cell exists**: Audits the loaded clean dataset to confirm label balances, text length distributions, and non-null guarantees before model initialization.

In [ ]:
print("==================================================================")
print("DATA AUDIT SUMMARY")
print("==================================================================")
print(f"Total Clean Dataset Size: {len(df):,} rows\n")

# Ticket Type Distribution
print("--- Ticket Type Distribution ---")
type_counts = df['ticket_type'].value_counts()
for t_type, count in type_counts.items():
    print(f"  {t_type:<15}: {count:,} ({count/len(df)*100:.2f}%)")

# Incident Module Distribution
df_inc = df[df['ticket_type'] == 'incident']
print("\n--- Incident Module Distribution ---")
mod_counts = df_inc['module'].value_counts()
for mod, count in mod_counts.items():
    print(f"  {mod:<10}: {count:,} ({count/len(df_inc)*100:.2f}%)")

# Issue Type Distribution
print("\n--- Incident Issue Type Distribution ---")
issue_counts = df_inc['issue_type'].value_counts()
for issue, count in issue_counts.items():
    print(f"  {issue:<20}: {count:,} ({count/len(df_inc)*100:.2f}%)")

# Text Length Statistics
word_counts = df['text'].astype(str).apply(lambda x: len(x.split()))
print("\n--- Word Count Statistics ---")
print(f"  Min words:    {word_counts.min()}")
print(f"  Max words:    {word_counts.max()}")
print(f"  Mean words:   {word_counts.mean():.2f}")
print(f"  Median words: {word_counts.median():.2f}")

# Error Code Percentage & Null Check
has_err_pct = (df['has_error_code'] == True).mean() * 100
null_text_count = df['text'].isna().sum()
print(f"\nTickets with Error Codes: {has_err_pct:.2f}%")
print(f"Null Text Values Found:    {null_text_count}")
print("==================================================================")

## 2.3 — Define Label Mappings and Export `label_mappings.json`

**Why this cell exists**: Establishes exact integer-to-category dictionaries required for model training and exports them to `label_mappings.json` for backend inference alignment.

In [ ]:
# Binary intake label mapping (0 = incident/edge_case, 1 = development)
BINARY_LABELS = {
    "incident":    0,
    "edge_case":   0,
    "development": 1,
}

# SAP Module label mappings
MODULE_LABELS = {
    "FI": 0, "MM": 1, "SD": 2, "PP": 3,
    "BASIS": 4, "ABAP": 5, "HR": 6,
}
MODULE_LABELS_INVERSE = {v: k for k, v in MODULE_LABELS.items()}

# Issue Type label mappings
ISSUE_TYPE_LABELS = {
    "Authorization": 0, "Configuration": 1, "Master Data": 2,
    "Performance": 3, "Integration": 4, "Data Inconsistency": 5,
    "User Error": 6, "System Error": 7,
}
ISSUE_TYPE_LABELS_INVERSE = {v: k for k, v in ISSUE_TYPE_LABELS.items()}

# Complexity mapping logic derived from issue_type
COMPLEXITY_MAP = {
    "Authorization":      "Simple",
    "User Error":         "Simple",
    "Master Data":        "Simple",
    "Configuration":      "Medium",
    "Data Inconsistency": "Medium",
    "Performance":        "Medium",
    "System Error":       "Complex",
    "Integration":        "Complex",
}
COMPLEXITY_LABELS = {"Simple": 0, "Medium": 1, "Complex": 2}
COMPLEXITY_LABELS_INVERSE = {v: k for k, v in COMPLEXITY_LABELS.items()}

# Export label mappings immediately to JSON file for backend reuse
LABEL_MAPPINGS_PATH = "label_mappings.json"
with open(LABEL_MAPPINGS_PATH, "w") as f:
    json.dump({
        "binary_labels":             BINARY_LABELS,
        "module_labels":             MODULE_LABELS,
        "module_labels_inverse":     MODULE_LABELS_INVERSE,
        "issue_type_labels":         ISSUE_TYPE_LABELS,
        "issue_type_labels_inverse": ISSUE_TYPE_LABELS_INVERSE,
        "complexity_map":            COMPLEXITY_MAP,
        "complexity_labels":         COMPLEXITY_LABELS,
        "complexity_labels_inverse": COMPLEXITY_LABELS_INVERSE,
    }, f, indent=2)

print(f"Successfully generated and saved label mappings to '{LABEL_MAPPINGS_PATH}'.")

## 2.4 — Prepare Binary Dataset and Stratified Splits (80 / 10 / 10)

**Why this cell exists**: Maps `binary_label` onto the full dataset and applies stratified splitting to guarantee identical class ratios across train (80%), validation (10%), and test (10%) sets.

In [ ]:
# Map binary labels onto tickets
df_binary = df[["text", "ticket_type"]].copy()
df_binary["binary_label"] = df_binary["ticket_type"].map(BINARY_LABELS)

# Drop unmapped rows if any
before_len = len(df_binary)
df_binary = df_binary.dropna(subset=["binary_label"])
df_binary["binary_label"] = df_binary["binary_label"].astype(int)
print(f"Dropped {before_len - len(df_binary)} unmapped binary rows.")
print(f"Binary dataset size: {len(df_binary):,}")
print(f"Class Distribution:\n{df_binary['binary_label'].value_counts()}\n")

# Execute Stratified 80/10/10 split
train_bin, temp_bin = train_test_split(
    df_binary, test_size=0.20, random_state=SEED, stratify=df_binary["binary_label"]
)
val_bin, test_bin = train_test_split(
    temp_bin, test_size=0.50, random_state=SEED, stratify=temp_bin["binary_label"]
)

print(f"Binary Train Set: {len(train_bin):,} rows ({len(train_bin)/len(df_binary)*100:.1f}%)")
print(f"Binary Val Set:   {len(val_bin):,} rows ({len(val_bin)/len(df_binary)*100:.1f}%)")
print(f"Binary Test Set:  {len(test_bin):,} rows ({len(test_bin)/len(df_binary)*100:.1f}%)")

## 2.5 — Prepare Multi-Label Dataset and Stratified Splits (80 / 10 / 10)

**Why this cell exists**: Filters dataset to incident and edge_case tickets only, maps module, issue_type, and complexity targets, and applies stratified splitting.

In [ ]:
# Filter dataset strictly to incident and edge_case tickets
df_multi = df[df["ticket_type"].isin(["incident", "edge_case"])].copy()
print(f"Multi-label dataset base size: {len(df_multi):,}")

# Apply all multi-task target label mappings
df_multi["module_label"] = df_multi["module"].map(MODULE_LABELS)
df_multi["issue_type_label"] = df_multi["issue_type"].map(ISSUE_TYPE_LABELS)
df_multi["complexity_label"] = df_multi["issue_type"].map(COMPLEXITY_MAP).map(COMPLEXITY_LABELS)

# Drop any unmapped rows
before_len = len(df_multi)
df_multi = df_multi.dropna(subset=["module_label", "issue_type_label", "complexity_label"])
df_multi[["module_label", "issue_type_label", "complexity_label"]] = \
    df_multi[["module_label", "issue_type_label", "complexity_label"]].astype(int)
print(f"Dropped {before_len - len(df_multi)} unmapped multi-label rows.")
print(f"Final multi-label dataset size: {len(df_multi):,}\n")

# Stratified 80/10/10 split based on module_label
train_multi, temp_multi = train_test_split(
    df_multi, test_size=0.20, random_state=SEED, stratify=df_multi["module_label"]
)
val_multi, test_multi = train_test_split(
    temp_multi, test_size=0.50, random_state=SEED, stratify=temp_multi["module_label"]
)

print(f"Multi-Label Train Set: {len(train_multi):,} rows ({len(train_multi)/len(df_multi)*100:.1f}%)")
print(f"Multi-Label Val Set:   {len(val_multi):,} rows ({len(val_multi)/len(df_multi)*100:.1f}%)")
print(f"Multi-Label Test Set:  {len(test_multi):,} rows ({len(test_multi)/len(df_multi)*100:.1f}%)")

# SECTION 3 — DISTILBERT FINE-TUNING

## 3.1 — DistilBERT Model Architecture Overview

**Why this section exists**: DistilBERT is a lightweight transformer model (67M parameters, 40% smaller than BERT-base) that retains 97% of BERT's language comprehension capabilities while running 60% faster, making it optimal for binary intake classification.

## 3.2 — Define DistilBERT Constants and File Paths

**Why this cell exists**: Declares model names, weights output paths, and directory structures as top-level variables.

In [ ]:
# Define DistilBERT model name and file paths
DISTILBERT_MODEL_NAME   = "distilbert-base-uncased"
DISTILBERT_WEIGHTS_PATH = "distilbert_intake_weights.pt"
DISTILBERT_MODEL_DIR    = "distilbert_intake_model"
DISTILBERT_ZIP_PATH     = "distilbert_intake_model.zip"
DISTILBERT_BEST_PATH    = "distilbert_best_checkpoint.pt"

print(f"DistilBERT model identifier: {DISTILBERT_MODEL_NAME}")

## 3.3 — Initialize Tokenizer and Pre-Trained DistilBERT Architecture

**Why this cell exists**: Loads the pre-trained DistilBERT tokenizer and sequence classification architecture (`num_labels=2`), moving weights onto GPU.

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Load DistilBERT tokenizer and classification model
distilbert_tokenizer = DistilBertTokenizer.from_pretrained(DISTILBERT_MODEL_NAME)
distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=2,
)
distilbert_model = distilbert_model.to(device)

# Calculate parameter counts
total_params = sum(p.numel() for p in distilbert_model.parameters())
trainable_params = sum(p.numel() for p in distilbert_model.parameters() if p.requires_grad)
print(f"DistilBERT Total Parameters:     {total_params:,}")
print(f"DistilBERT Trainable Parameters: {trainable_params:,}")
print(f"Model assigned to: {device}")

## 3.4 — Construct PyTorch Dataset and DataLoader Objects

**Why this cell exists**: Implements `SAPBinaryDataset` for dynamic tokenization (`max_length=128`) and encapsulates datasets into PyTorch DataLoaders with `batch_size=32` (train) and `64` (eval).

In [ ]:
class SAPBinaryDataset(Dataset):
    """
    Custom PyTorch Dataset for DistilBERT binary classification.
    Performs tokenization dynamically on item retrieval.
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts.tolist()
        self.labels     = labels.tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_length,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Define DataLoader batch sizes
DISTILBERT_BATCH_TRAIN = 32
DISTILBERT_BATCH_EVAL  = 64

# Instantiate PyTorch Dataset instances
train_bin_dataset = SAPBinaryDataset(train_bin["text"], train_bin["binary_label"], distilbert_tokenizer)
val_bin_dataset   = SAPBinaryDataset(val_bin["text"],   val_bin["binary_label"],   distilbert_tokenizer)
test_bin_dataset  = SAPBinaryDataset(test_bin["text"],  test_bin["binary_label"],  distilbert_tokenizer)

# Instantiate PyTorch DataLoaders
train_bin_loader = DataLoader(train_bin_dataset, batch_size=DISTILBERT_BATCH_TRAIN, shuffle=True,  num_workers=2)
val_bin_loader   = DataLoader(val_bin_dataset,   batch_size=DISTILBERT_BATCH_EVAL,  shuffle=False, num_workers=2)
test_bin_loader  = DataLoader(test_bin_dataset,  batch_size=DISTILBERT_BATCH_EVAL,  shuffle=False, num_workers=2)

print(f"Binary Train Batches: {len(train_bin_loader)} | Val Batches: {len(val_bin_loader)} | Test Batches: {len(test_bin_loader)}")

## 3.5 — Configure Hyperparameters, Optimizer, and Learning Rate Scheduler

**Why this cell exists**: Sets AdamW optimizer (`lr=2e-5`, `weight_decay=0.01`), linear warmup schedule over 100 steps, and CrossEntropyLoss criterion.

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

# DistilBERT Hyperparameters
DISTILBERT_EPOCHS    = 3
DISTILBERT_LR        = 2e-5
DISTILBERT_WARMUP    = 100
DISTILBERT_WD        = 0.01

# Instantiate AdamW optimizer with weight decay regularization
optimizer_distilbert = AdamW(
    distilbert_model.parameters(),
    lr           = DISTILBERT_LR,
    weight_decay = DISTILBERT_WD,
)

# Set linear warmup scheduler over total training iterations
total_steps_distilbert = len(train_bin_loader) * DISTILBERT_EPOCHS
scheduler_distilbert = get_linear_schedule_with_warmup(
    optimizer_distilbert,
    num_warmup_steps   = DISTILBERT_WARMUP,
    num_training_steps = total_steps_distilbert,
)

criterion_distilbert = nn.CrossEntropyLoss()
print(f"DistilBERT Configured: Epochs={DISTILBERT_EPOCHS}, LR={DISTILBERT_LR}, Total Steps={total_steps_distilbert}")

## 3.6 — Execute DistilBERT Training Loop and Early Checkpointing

**Why this cell exists**: Fine-tunes DistilBERT across 3 epochs, logs loss and F1 metrics, and saves the best model checkpoint based on lowest validation loss.

In [ ]:
# Track metrics across training epochs
distilbert_history = {
    "train_loss": [], "val_loss": [],
    "val_accuracy": [], "val_f1_weighted": [], "val_f1_macro": []
}
best_val_loss_distilbert = float("inf")

for epoch in range(DISTILBERT_EPOCHS):
    # --- TRAINING PHASE ---
    distilbert_model.train()
    running_train_loss = 0.0
    
    progress_bar = tqdm(train_bin_loader, desc=f"Epoch {epoch+1}/{DISTILBERT_EPOCHS} [Train]")
    for step, batch in enumerate(progress_bar):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)
        
        optimizer_distilbert.zero_grad()
        outputs = distilbert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(distilbert_model.parameters(), max_norm=1.0)
        
        optimizer_distilbert.step()
        scheduler_distilbert.step()
        
        running_train_loss += loss.item()
        progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
        
    avg_train_loss = running_train_loss / len(train_bin_loader)
    
    # --- VALIDATION PHASE ---
    distilbert_model.eval()
    running_val_loss = 0.0
    val_preds, val_targets = [], []
    
    with torch.no_grad():
        for batch in tqdm(val_bin_loader, desc=f"Epoch {epoch+1}/{DISTILBERT_EPOCHS} [Val]"):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)
            
            outputs = distilbert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            running_val_loss += outputs.loss.item()
            
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = running_val_loss / len(val_bin_loader)
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1_weighted = f1_score(val_targets, val_preds, average="weighted")
    val_f1_macro    = f1_score(val_targets, val_preds, average="macro")
    
    # Store metrics history
    distilbert_history["train_loss"].append(avg_train_loss)
    distilbert_history["val_loss"].append(avg_val_loss)
    distilbert_history["val_accuracy"].append(val_acc)
    distilbert_history["val_f1_weighted"].append(val_f1_weighted)
    distilbert_history["val_f1_macro"].append(val_f1_macro)
    
    print(f"\nEpoch {epoch+1}/{DISTILBERT_EPOCHS} Summary:")
    print(f"  Train Loss:         {avg_train_loss:.4f}")
    print(f"  Val Loss:           {avg_val_loss:.4f}")
    print(f"  Val Accuracy:       {val_acc:.4f}")
    print(f"  Val F1 (Weighted):  {val_f1_weighted:.4f}")
    print(f"  Val F1 (Macro):     {val_f1_macro:.4f}")
    
    # Save best model checkpoint
    if avg_val_loss < best_val_loss_distilbert:
        best_val_loss_distilbert = avg_val_loss
        torch.save(distilbert_model.state_dict(), DISTILBERT_BEST_PATH)
        print(f"  -> Checkpoint saved to '{DISTILBERT_BEST_PATH}' (Val Loss: {avg_val_loss:.4f})\n")

## 3.7 — Plot DistilBERT Training Curves

**Why this cell exists**: Visualizes loss trajectories and validation performance over training epochs to verify convergence.

In [ ]:
# Render side-by-side training charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, DISTILBERT_EPOCHS + 1)

# Loss Plot
ax1.plot(epochs_range, distilbert_history["train_loss"], 'o-', label="Train Loss", linewidth=2)
ax1.plot(epochs_range, distilbert_history["val_loss"],   's-', label="Val Loss",   linewidth=2)
ax1.set_title("DistilBERT Loss Trajectory", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Cross Entropy Loss", fontsize=11)
ax1.legend()

# Metrics Plot
ax2.plot(epochs_range, distilbert_history["val_accuracy"],    '^--', label="Val Accuracy", linewidth=2)
ax2.plot(epochs_range, distilbert_history["val_f1_weighted"], 'd-',  label="Val F1 Weighted", linewidth=2)
ax2.set_title("DistilBERT Validation Performance", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Score", fontsize=11)
ax2.legend()

plt.tight_layout()
plt.show()

## 3.8 — DistilBERT Unbiased Test Set Evaluation

**Why this cell exists**: Evaluates the best DistilBERT checkpoint once on the held-out test set (10% hidden data) and plots a confusion matrix.

In [ ]:
# Load best checkpoint for test set evaluation
distilbert_model.load_state_dict(torch.load(DISTILBERT_BEST_PATH))
distilbert_model.eval()

test_preds, test_targets = [], []
with torch.no_grad():
    for batch in tqdm(test_bin_loader, desc="Evaluating DistilBERT Test Set"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)
        
        outputs = distilbert_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_targets.extend(labels.cpu().numpy())

# Calculate evaluation metrics
distilbert_test_acc          = accuracy_score(test_targets, test_preds)
distilbert_test_f1_weighted  = f1_score(test_targets, test_preds, average="weighted")
distilbert_test_f1_macro     = f1_score(test_targets, test_preds, average="macro")
distilbert_test_prec_weighted= precision_score(test_targets, test_preds, average="weighted")
distilbert_test_rec_weighted = recall_score(test_targets, test_preds, average="weighted")

print("\n" + "═" * 55)
print("DISTILBERT TEST SET RESULTS")
print("═" * 55)
print(f"Accuracy:           {distilbert_test_acc:.4f}")
print(f"F1 Weighted:        {distilbert_test_f1_weighted:.4f}")
print(f"F1 Macro:           {distilbert_test_f1_macro:.4f}")
print(f"Precision Weighted: {distilbert_test_prec_weighted:.4f}")
print(f"Recall Weighted:    {distilbert_test_rec_weighted:.4f}\n")
print("Per-Class Classification Report:")
print(classification_report(test_targets, test_preds, target_names=["incident", "development"]))

# Plot Confusion Matrix Heatmap
cm = confusion_matrix(test_targets, test_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", xticklabels=["Incident (0)", "Development (1)"], yticklabels=["Incident (0)", "Development (1)"], ax=ax)
ax.set_title("DistilBERT Confusion Matrix — Test Set", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.show()

# Target Benchmark Check
if distilbert_test_f1_weighted >= 0.85:
    print(f"TARGET MET: F1 Weighted {distilbert_test_f1_weighted:.4f} >= 0.85 threshold.")
else:
    print(f"WARNING: F1 Weighted {distilbert_test_f1_weighted:.4f} is below 0.85 threshold.")

## 3.9 — Persist DistilBERT Model Weights and Download Package

**Why this cell exists**: Saves model and tokenizer artifacts into `distilbert_intake_model/` and zips them for automatic browser download.

In [ ]:
# Save pretrained model and tokenizer directory
distilbert_model.load_state_dict(torch.load(DISTILBERT_BEST_PATH))
distilbert_model.save_pretrained(DISTILBERT_MODEL_DIR)
distilbert_tokenizer.save_pretrained(DISTILBERT_MODEL_DIR)
torch.save(distilbert_model.state_dict(), DISTILBERT_WEIGHTS_PATH)

# Archive model directory into zip format
shutil.make_archive(DISTILBERT_MODEL_DIR, "zip", DISTILBERT_MODEL_DIR)

# Trigger browser file downloads in Colab
files.download(DISTILBERT_ZIP_PATH)
files.download(DISTILBERT_WEIGHTS_PATH)

print("Successfully downloaded DistilBERT package:")
print(f"  1. {DISTILBERT_ZIP_PATH} -> Extract into backend/models/distilbert_intake/")
print(f"  2. {DISTILBERT_WEIGHTS_PATH} -> Copy into backend/models/distilbert_intake/")

# SECTION 4 — BERT MULTI-LABEL FINE-TUNING

## 4.1 — Multi-Task Learning Architecture Overview

**Why this section exists**: Explains multi-task learning where a single shared `bert-base-uncased` backbone (110M parameters) feeds into three independent linear classification heads (`module_head`, `issue_type_head`, `complexity_head`), training all three tasks concurrently.

## 4.2 — Define BERT Constants and File Paths

**Why this cell exists**: Sets identifier string, weights filename, tokenizer path, and config JSON path variables.

In [ ]:
# Define BERT model parameters and file paths
BERT_MODEL_NAME      = "bert-base-uncased"
BERT_WEIGHTS_PATH    = "bert_classifier_weights.pt"
BERT_TOKENIZER_DIR   = "bert_classifier_tokenizer"
BERT_TOKENIZER_ZIP   = "bert_classifier_tokenizer.zip"
BERT_CONFIG_PATH     = "bert_classifier_config.json"
BERT_BEST_PATH       = "bert_best_checkpoint.pt"

print(f"BERT model identifier: {BERT_MODEL_NAME}")

## 4.3 — Custom Multi-Task BERT Architecture Implementation

**Why this cell exists**: Implements `BERTMultiLabelClassifier` containing a shared BERT backbone and three separate dense output heads attached to the `[CLS]` token output.

In [ ]:
from transformers import BertModel

class BERTMultiLabelClassifier(nn.Module):
    """
    Custom Multi-Task BERT Model.
    Extracts the [CLS] token vector (768-dim) and passes it through
    three parallel classification heads for module, issue_type, and complexity.
    """
    def __init__(self, bert_model_name, num_modules, num_issue_types, num_complexities, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden_size = self.bert.config.hidden_size  # 768
        
        self.dropout         = nn.Dropout(dropout)
        self.module_head     = nn.Linear(hidden_size, num_modules)
        self.issue_type_head = nn.Linear(hidden_size, num_issue_types)
        self.complexity_head = nn.Linear(hidden_size, num_complexities)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # Extract [CLS] token representation
        cls_output = self.dropout(cls_output)
        
        module_logits     = self.module_head(cls_output)
        issue_type_logits = self.issue_type_head(cls_output)
        complexity_logits = self.complexity_head(cls_output)
        
        return module_logits, issue_type_logits, complexity_logits

# Instantiate multi-task model architecture
bert_model = BERTMultiLabelClassifier(
    bert_model_name   = BERT_MODEL_NAME,
    num_modules       = len(MODULE_LABELS),
    num_issue_types   = len(ISSUE_TYPE_LABELS),
    num_complexities  = len(COMPLEXITY_LABELS),
    dropout           = 0.3,
).to(device)

# Calculate parameter counts
total_params_bert = sum(p.numel() for p in bert_model.parameters())
trainable_params_bert = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"BERT Multi-Task Total Parameters:     {total_params_bert:,}")
print(f"BERT Multi-Task Trainable Parameters: {trainable_params_bert:,}")

## 4.4 — Multi-Label Dataset and DataLoader Construction

**Why this cell exists**: Implements `SAPMultiLabelDataset` to return tokenized text along with three target label tensors, setting batch sizes to `16` (train) and `32` (eval).

In [ ]:
from transformers import BertTokenizer
bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

class SAPMultiLabelDataset(Dataset):
    """
    Custom PyTorch Dataset for Multi-Label BERT Classification.
    """
    def __init__(self, texts, module_labels, issue_type_labels, complexity_labels, tokenizer, max_length=128):
        self.texts             = texts.tolist()
        self.module_labels     = module_labels.tolist()
        self.issue_type_labels = issue_type_labels.tolist()
        self.complexity_labels = complexity_labels.tolist()
        self.tokenizer         = tokenizer
        self.max_length        = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_length,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        return {
            "input_ids":        encoding["input_ids"].squeeze(0),
            "attention_mask":   encoding["attention_mask"].squeeze(0),
            "module_label":     torch.tensor(self.module_labels[idx],     dtype=torch.long),
            "issue_type_label": torch.tensor(self.issue_type_labels[idx], dtype=torch.long),
            "complexity_label": torch.tensor(self.complexity_labels[idx], dtype=torch.long),
        }

# Define BERT batch sizes (16 for train to manage 110M param VRAM footprint)
BERT_BATCH_TRAIN = 16
BERT_BATCH_EVAL  = 32

# Instantiate Datasets
train_multi_dataset = SAPMultiLabelDataset(
    train_multi["text"], train_multi["module_label"], train_multi["issue_type_label"], train_multi["complexity_label"], bert_tokenizer
)
val_multi_dataset = SAPMultiLabelDataset(
    val_multi["text"], val_multi["module_label"], val_multi["issue_type_label"], val_multi["complexity_label"], bert_tokenizer
)
test_multi_dataset = SAPMultiLabelDataset(
    test_multi["text"], test_multi["module_label"], test_multi["issue_type_label"], test_multi["complexity_label"], bert_tokenizer
)

# Instantiate DataLoaders
train_multi_loader = DataLoader(train_multi_dataset, batch_size=BERT_BATCH_TRAIN, shuffle=True,  num_workers=2)
val_multi_loader   = DataLoader(val_multi_dataset,   batch_size=BERT_BATCH_EVAL,  shuffle=False, num_workers=2)
test_multi_loader  = DataLoader(test_multi_dataset,  batch_size=BERT_BATCH_EVAL,  shuffle=False, num_workers=2)

print(f"Multi-Label Train Batches: {len(train_multi_loader)} | Val Batches: {len(val_multi_loader)} | Test Batches: {len(test_multi_loader)}")

## 4.5 — BERT Multi-Task Loss Functions and Training Configuration

**Why this cell exists**: Sets up three separate CrossEntropyLoss criteria, AdamW optimizer (`lr=2e-5`), and linear warmup schedule over 4 epochs.

In [ ]:
# BERT Hyperparameters
BERT_EPOCHS  = 4
BERT_LR      = 2e-5
BERT_WARMUP  = 200
BERT_WD      = 0.01

# Three CrossEntropyLoss criteria for equal task weighting
criterion_module      = nn.CrossEntropyLoss()
criterion_issue_type  = nn.CrossEntropyLoss()
criterion_complexity  = nn.CrossEntropyLoss()

# Optimizer & Scheduler
optimizer_bert = AdamW(bert_model.parameters(), lr=BERT_LR, weight_decay=BERT_WD)
total_steps_bert = len(train_multi_loader) * BERT_EPOCHS
scheduler_bert = get_linear_schedule_with_warmup(
    optimizer_bert,
    num_warmup_steps   = BERT_WARMUP,
    num_training_steps = total_steps_bert,
)

print(f"BERT Multi-Task Configured: Epochs={BERT_EPOCHS}, LR={BERT_LR}, Total Steps={total_steps_bert}")

## 4.6 — Execute BERT Multi-Task Training Loop and Checkpointing

**Why this cell exists**: Runs joint training by summing task losses (`loss = loss_mod + loss_issue + loss_comp`), evaluates per-task validation F1 scores, and saves the best checkpoint based on average validation F1.

In [ ]:
# Track BERT training statistics
bert_history = {
    "train_loss": [], "val_loss": [],
    "val_module_f1": [], "val_issue_type_f1": [],
    "val_complexity_f1": [], "val_avg_f1": [],
}
best_avg_f1_bert = 0.0

for epoch in range(BERT_EPOCHS):
    # --- TRAINING PHASE ---
    bert_model.train()
    running_train_loss = 0.0
    
    progress_bar = tqdm(train_multi_loader, desc=f"Epoch {epoch+1}/{BERT_EPOCHS} [Train]")
    for batch in progress_bar:
        input_ids        = batch["input_ids"].to(device)
        attention_mask   = batch["attention_mask"].to(device)
        module_labels    = batch["module_label"].to(device)
        issue_labels     = batch["issue_type_label"].to(device)
        comp_labels      = batch["complexity_label"].to(device)
        
        optimizer_bert.zero_grad()
        mod_logits, issue_logits, comp_logits = bert_model(input_ids, attention_mask)
        
        loss_mod   = criterion_module(mod_logits, module_labels)
        loss_issue = criterion_issue_type(issue_logits, issue_labels)
        loss_comp  = criterion_complexity(comp_logits, comp_labels)
        
        # Sum multi-task losses
        combined_loss = loss_mod + loss_issue + loss_comp
        combined_loss.backward()
        nn.utils.clip_grad_norm_(bert_model.parameters(), max_norm=1.0)
        
        optimizer_bert.step()
        scheduler_bert.step()
        
        running_train_loss += combined_loss.item()
        progress_bar.set_postfix({"joint_loss": f"{combined_loss.item():.4f}"})
        
    avg_train_loss = running_train_loss / len(train_multi_loader)
    
    # --- VALIDATION PHASE ---
    bert_model.eval()
    running_val_loss = 0.0
    mod_preds, mod_targets = [], []
    issue_preds, issue_targets = [], []
    comp_preds, comp_targets = [], []
    
    with torch.no_grad():
        for batch in tqdm(val_multi_loader, desc=f"Epoch {epoch+1}/{BERT_EPOCHS} [Val]"):
            input_ids        = batch["input_ids"].to(device)
            attention_mask   = batch["attention_mask"].to(device)
            module_labels    = batch["module_label"].to(device)
            issue_labels     = batch["issue_type_label"].to(device)
            comp_labels      = batch["complexity_label"].to(device)
            
            mod_logits, issue_logits, comp_logits = bert_model(input_ids, attention_mask)
            
            l_mod   = criterion_module(mod_logits, module_labels)
            l_issue = criterion_issue_type(issue_logits, issue_labels)
            l_comp  = criterion_complexity(comp_logits, comp_labels)
            running_val_loss += (l_mod + l_issue + l_comp).item()
            
            mod_preds.extend(torch.argmax(mod_logits, dim=1).cpu().numpy())
            mod_targets.extend(module_labels.cpu().numpy())
            
            issue_preds.extend(torch.argmax(issue_logits, dim=1).cpu().numpy())
            issue_targets.extend(issue_labels.cpu().numpy())
            
            comp_preds.extend(torch.argmax(comp_logits, dim=1).cpu().numpy())
            comp_targets.extend(comp_labels.cpu().numpy())
            
    avg_val_loss = running_val_loss / len(val_multi_loader)
    val_mod_f1   = f1_score(mod_targets, mod_preds, average="weighted")
    val_issue_f1 = f1_score(issue_targets, issue_preds, average="weighted")
    val_comp_f1  = f1_score(comp_targets, comp_preds, average="weighted")
    val_avg_f1   = (val_mod_f1 + val_issue_f1 + val_comp_f1) / 3.0
    
    bert_history["train_loss"].append(avg_train_loss)
    bert_history["val_loss"].append(avg_val_loss)
    bert_history["val_module_f1"].append(val_mod_f1)
    bert_history["val_issue_type_f1"].append(val_issue_f1)
    bert_history["val_complexity_f1"].append(val_comp_f1)
    bert_history["val_avg_f1"].append(val_avg_f1)
    
    print(f"\nEpoch {epoch+1}/{BERT_EPOCHS} Summary:")
    print(f"  Train Loss:         {avg_train_loss:.4f}")
    print(f"  Val Loss:           {avg_val_loss:.4f}")
    print(f"  Val Module F1:      {val_mod_f1:.4f}")
    print(f"  Val Issue Type F1:  {val_issue_f1:.4f}")
    print(f"  Val Complexity F1:  {val_comp_f1:.4f}")
    print(f"  Val Average F1:     {val_avg_f1:.4f}")
    
    # Save best model checkpoint based on average validation F1
    if val_avg_f1 > best_avg_f1_bert:
        best_avg_f1_bert = val_avg_f1
        torch.save(bert_model.state_dict(), BERT_BEST_PATH)
        print(f"  -> Checkpoint saved to '{BERT_BEST_PATH}' (Val Avg F1: {val_avg_f1:.4f})\n")

## 4.7 — Plot BERT Multi-Task Training Curves

**Why this cell exists**: Plots loss trajectory and task-specific F1 score curves across 3 subplot rows.

In [ ]:
# Render multi-panel training plots
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
epochs_range = range(1, BERT_EPOCHS + 1)

# Subplot 1: Loss
ax1.plot(epochs_range, bert_history["train_loss"], 'o-', label="Train Combined Loss", linewidth=2)
ax1.plot(epochs_range, bert_history["val_loss"],   's-', label="Val Combined Loss",   linewidth=2)
ax1.set_title("BERT Joint Loss Trajectory", fontsize=13, fontweight="bold")
ax1.set_ylabel("Loss", fontsize=11)
ax1.legend()

# Subplot 2: Module & Issue F1
ax2.plot(epochs_range, bert_history["val_module_f1"],     '^--', label="Module Val F1",     linewidth=2)
ax2.plot(epochs_range, bert_history["val_issue_type_f1"], 'd--', label="Issue Type Val F1", linewidth=2)
ax2.set_title("BERT Module and Issue Type Validation F1", fontsize=13, fontweight="bold")
ax2.set_ylabel("F1 Score", fontsize=11)
ax2.legend()

# Subplot 3: Complexity & Average F1
ax3.plot(epochs_range, bert_history["val_complexity_f1"], 'x--', label="Complexity Val F1", linewidth=2)
ax3.plot(epochs_range, bert_history["val_avg_f1"],        '*-',  label="Average Val F1",    linewidth=2.5, color="purple")
ax3.set_title("BERT Complexity and Average Validation F1", fontsize=13, fontweight="bold")
ax3.set_xlabel("Epoch", fontsize=11)
ax3.set_ylabel("F1 Score", fontsize=11)
ax3.legend()

plt.tight_layout()
plt.show()

## 4.8 — BERT Multi-Task Unbiased Test Set Evaluation

**Why this cell exists**: Runs inference on the test set, prints classification reports for all three heads, plots 3 confusion matrices, and assesses target benchmarks.

In [ ]:
# Load best BERT checkpoint
bert_model.load_state_dict(torch.load(BERT_BEST_PATH))
bert_model.eval()

mod_preds, mod_targets = [], []
issue_preds, issue_targets = [], []
comp_preds, comp_targets = [], []

with torch.no_grad():
    for batch in tqdm(test_multi_loader, desc="Evaluating BERT Multi-Task Test Set"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        m_logits, i_logits, c_logits = bert_model(input_ids, attention_mask)
        
        mod_preds.extend(torch.argmax(m_logits, dim=1).cpu().numpy())
        mod_targets.extend(batch["module_label"].numpy())
        
        issue_preds.extend(torch.argmax(i_logits, dim=1).cpu().numpy())
        issue_targets.extend(batch["issue_type_label"].numpy())
        
        comp_preds.extend(torch.argmax(c_logits, dim=1).cpu().numpy())
        comp_targets.extend(batch["complexity_label"].numpy())

# Compute Task Metrics
module_accuracy    = accuracy_score(mod_targets, mod_preds)
module_f1_weighted = f1_score(mod_targets, mod_preds, average="weighted")
module_f1_macro    = f1_score(mod_targets, mod_preds, average="macro")

issue_accuracy     = accuracy_score(issue_targets, issue_preds)
issue_f1_weighted  = f1_score(issue_targets, issue_preds, average="weighted")
issue_f1_macro     = f1_score(issue_targets, issue_preds, average="macro")

complex_accuracy   = accuracy_score(comp_targets, comp_preds)
complex_f1_weighted= f1_score(comp_targets, comp_preds, average="weighted")
complex_f1_macro   = f1_score(comp_targets, comp_preds, average="macro")

overall_avg_f1     = (module_f1_weighted + issue_f1_weighted + complex_f1_weighted) / 3.0

print("═" * 55)
print("BERT MULTI-LABEL TEST SET RESULTS")
print("═" * 55)

print("\nMODULE CLASSIFICATION (7 classes):")
print(f"  Accuracy:    {module_accuracy:.4f}")
print(f"  F1 Weighted: {module_f1_weighted:.4f}")
print(f"  F1 Macro:    {module_f1_macro:.4f}")
print(classification_report(mod_targets, mod_preds, target_names=list(MODULE_LABELS.keys())))

print("\nISSUE TYPE CLASSIFICATION (8 classes):")
print(f"  Accuracy:    {issue_accuracy:.4f}")
print(f"  F1 Weighted: {issue_f1_weighted:.4f}")
print(f"  F1 Macro:    {issue_f1_macro:.4f}")
print(classification_report(issue_targets, issue_preds, target_names=list(ISSUE_TYPE_LABELS.keys())))

print("\nCOMPLEXITY CLASSIFICATION (3 classes):")
print(f"  Accuracy:    {complex_accuracy:.4f}")
print(f"  F1 Weighted: {complex_f1_weighted:.4f}")
print(f"  F1 Macro:    {complex_f1_macro:.4f}")
print(classification_report(comp_targets, comp_preds, target_names=list(COMPLEXITY_LABELS.keys())))

print(f"\nOVERALL AVERAGE F1 SCORE: {overall_avg_f1:.4f}\n")

# Render 3 Confusion Matrices Side-by-Side
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(confusion_matrix(mod_targets, mod_preds), annot=True, fmt=",d", cmap="Purples", xticklabels=list(MODULE_LABELS.keys()), yticklabels=list(MODULE_LABELS.keys()), ax=ax1)
ax1.set_title("Module Confusion Matrix", fontsize=12, fontweight="bold")
ax1.set_xlabel("Predicted")
ax1.set_ylabel("True")

sns.heatmap(confusion_matrix(issue_targets, issue_preds), annot=True, fmt=",d", cmap="Greens", xticklabels=list(ISSUE_TYPE_LABELS.keys()), yticklabels=list(ISSUE_TYPE_LABELS.keys()), ax=ax2)
ax2.set_title("Issue Type Confusion Matrix", fontsize=12, fontweight="bold")
ax2.set_xlabel("Predicted")
ax2.tick_params(axis='x', rotation=45)

sns.heatmap(confusion_matrix(comp_targets, comp_preds), annot=True, fmt=",d", cmap="Oranges", xticklabels=list(COMPLEXITY_LABELS.keys()), yticklabels=list(COMPLEXITY_LABELS.keys()), ax=ax3)
ax3.set_title("Complexity Confusion Matrix", fontsize=12, fontweight="bold")
ax3.set_xlabel("Predicted")

plt.tight_layout()
plt.show()

# Benchmark Target Checks
print("\n--- Target Benchmark Status ---")
if module_f1_weighted >= 0.80:
    print(f"  MODULE TARGET MET: F1 Weighted {module_f1_weighted:.4f} >= 0.80")
else:
    print(f"  WARNING: Module F1 Weighted {module_f1_weighted:.4f} < 0.80")

if issue_f1_weighted >= 0.75:
    print(f"  ISSUE TYPE TARGET MET: F1 Weighted {issue_f1_weighted:.4f} >= 0.75")
else:
    print(f"  WARNING: Issue Type F1 Weighted {issue_f1_weighted:.4f} < 0.75")

if complex_f1_weighted >= 0.70:
    print(f"  COMPLEXITY TARGET MET: F1 Weighted {complex_f1_weighted:.4f} >= 0.70")
else:
    print(f"  WARNING: Complexity F1 Weighted {complex_f1_weighted:.4f} < 0.70")

## 4.9 — Persist BERT Model Weights and Configuration Package

**Why this cell exists**: Saves state dict to `bert_classifier_weights.pt`, tokenizer to `bert_classifier_tokenizer/` (zipped), and writes `bert_classifier_config.json`.

In [ ]:
# Load best BERT model weights
bert_model.load_state_dict(torch.load(BERT_BEST_PATH))
torch.save(bert_model.state_dict(), BERT_WEIGHTS_PATH)
bert_tokenizer.save_pretrained(BERT_TOKENIZER_DIR)

# Create config dict JSON for model instantiation in backend
bert_config_dict = {
    "bert_model_name":  BERT_MODEL_NAME,
    "num_modules":      len(MODULE_LABELS),
    "num_issue_types":  len(ISSUE_TYPE_LABELS),
    "num_complexities": len(COMPLEXITY_LABELS),
    "dropout":          0.3,
    "max_length":       128,
}
with open(BERT_CONFIG_PATH, "w") as f:
    json.dump(bert_config_dict, f, indent=2)

# Archive tokenizer directory into zip format
shutil.make_archive(BERT_TOKENIZER_DIR, "zip", BERT_TOKENIZER_DIR)

# Download files to local environment
files.download(BERT_WEIGHTS_PATH)
files.download(BERT_TOKENIZER_ZIP)
files.download(BERT_CONFIG_PATH)

print("Successfully downloaded BERT package:")
print(f"  1. {BERT_WEIGHTS_PATH} -> Copy to backend/models/bert_classifier/")
print(f"  2. {BERT_TOKENIZER_ZIP} -> Extract into backend/models/bert_classifier/tokenizer/")
print(f"  3. {BERT_CONFIG_PATH} -> Copy to backend/models/bert_classifier/")

# SECTION 5 — DOWNLOAD SHARED FILES AND SUMMARY

## 5.1 — Download Label Mappings JSON

**Why this cell exists**: Triggers direct browser download of `label_mappings.json`.

In [ ]:
# Download label_mappings.json
files.download(LABEL_MAPPINGS_PATH)
print(f"Downloaded '{LABEL_MAPPINGS_PATH}'. Place into 'backend/models/'.")

## 5.2 — Final Training Summary and Backend Deployment Guide

**Why this cell exists**: Prints an overall summary of test metrics and clear directory instructions for deploying fine-tuned models into the FastAPI backend.

In [ ]:
print("═" * 55)
print("TRAINING COMPLETE — FINAL SUMMARY REPORT")
print("═" * 55)
print()
print("1. DistilBERT Intake Classifier (Binary: Incident vs Development):")
print(f"   - Test Accuracy:          {distilbert_test_acc:.4f}")
print(f"   - Test F1 (Weighted):     {distilbert_test_f1_weighted:.4f}")
print(f"   - Test F1 (Macro):        {distilbert_test_f1_macro:.4f}")
print()
print("2. BERT Multi-Task Classifier (Incident Processing):")
print(f"   - Module F1 (Weighted):     {module_f1_weighted:.4f}")
print(f"   - Issue Type F1 (Weighted): {issue_f1_weighted:.4f}")
print(f"   - Complexity F1 (Weighted): {complex_f1_weighted:.4f}")
print(f"   - Overall Average F1:       {overall_avg_f1:.4f}")
print()
print("=========================================================")
print("BACKEND FILE PLACEMENT INSTRUCTIONS:")
print("  Move downloaded artifacts into backend/models/ directory:")
print("    1. Extract distilbert_intake_model.zip into:")
print("       backend/models/distilbert_intake/")
print("    2. Move distilbert_intake_weights.pt into:")
print("       backend/models/distilbert_intake/")
print("    3. Extract bert_classifier_tokenizer.zip into:")
print("       backend/models/bert_classifier/tokenizer/")
print("    4. Move bert_classifier_weights.pt and bert_classifier_config.json into:")
print("       backend/models/bert_classifier/")
print("    5. Move label_mappings.json into:")
print("       backend/models/label_mappings.json")
print("=========================================================")